# Module 5: Agent Frameworks
# Topics 33 & 34: Chains & Runnables

> **Interview Difficulty:** ⭐⭐⭐⭐⭐ (Must Know)
>
> **Interview Frequency:** Extremely High
>
> **Prerequisites:**
> - Models ✅
> - Prompt Templates ✅
> - Messages ✅
> - Output Parsers ✅
> - LCEL ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- What is a Chain?
- What is a Runnable?
- Difference between Chain and Runnable
- Types of Runnables
- RunnableSequence
- RunnableLambda
- RunnableParallel
- RunnableBranch
- RunnablePassthrough
- RunnableAssign
- How everything fits together

---

# 1. What is a Chain?

A **Chain** is a workflow where the output of one step becomes the input of the next step.

Simple example:

```text
Question

↓

Prompt

↓

LLM

↓

Parser

↓

Answer
```

This entire workflow is called a **Chain**.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **A Chain is a sequence of operations where data flows through multiple LangChain components to accomplish a task.**

---

# 2. Traditional Chains (Older LangChain)

Earlier versions of LangChain had dedicated chain classes.

Examples:

- LLMChain
- SequentialChain
- ConversationChain
- RetrievalQA

Example:

```python
chain = LLMChain(
    llm=llm,
    prompt=prompt
)
```

---

### Why were they replaced?

Problems:

- Too many chain classes
- Difficult to combine
- Less flexible
- Hard to extend

---

# 3. Modern Chains (LCEL)

Today, chains are created using LCEL.

Instead of

```python
LLMChain(...)
```

we write

```python
chain = prompt | llm
```

or

```python
chain = prompt | llm | parser
```

Much simpler.

---

# 4. Chain Execution

```text
Input

↓

Prompt

↓

Messages

↓

LLM

↓

AIMessage

↓

Parser

↓

Output
```

Every step executes automatically.

---

# 5. Real Example

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Explain {topic}."
)

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke(
    {
        "topic": "LangChain"
    }
)

print(result)
```

---

# 6. What is a Runnable?

Everything in modern LangChain is built around **Runnables**.

Examples:

```text
Prompt

↓

Runnable

Chat Model

↓

Runnable

Output Parser

↓

Runnable
```

Each component follows the same interface.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **A Runnable is any executable LangChain component that implements standardized methods such as invoke(), batch(), stream(), and ainvoke().**

---

# 7. Runnable Interface

Every Runnable supports:

```python
invoke()
```

Single request

---

```python
batch()
```

Multiple requests

---

```python
stream()
```

Streaming output

---

```python
ainvoke()
```

Async execution

---

This common interface makes different components interchangeable.

---

# 8. RunnableSequence

This is what happens internally when you write:

```python
prompt | llm | parser
```

Internally:

```text
RunnableSequence

│

├── Prompt

├── LLM

└── Parser
```

So:

```python
prompt | llm | parser
```

is equivalent to:

```python
RunnableSequence(
    prompt,
    llm,
    parser
)
```

The `|` operator is syntactic sugar for building a `RunnableSequence`.

---

# 9. RunnableLambda

Allows custom Python code inside a chain.

Example

```python
from langchain_core.runnables import RunnableLambda

def to_upper(text):
    return text.upper()

uppercase = RunnableLambda(to_upper)

print(
    uppercase.invoke("hello")
)
```

Output

```
HELLO
```

---

Chain

```python
chain = (
    prompt
    | llm
    | uppercase
)
```

---

# 10. RunnableParallel

Runs multiple tasks simultaneously.

Example

```text
User Question

        │

        ▼

RunnableParallel

   │           │

   ▼           ▼

Summary    Keywords

   │           │

   └──────┬──────┘

          ▼

Combined Result
```

Example

```python
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain
)

result = parallel.invoke(
    "Explain LangChain"
)
```

Output

```python
{
    "summary": "...",
    "keywords": "..."
}
```

---

# 11. RunnableBranch

Conditional execution.

Example

```text
Question

↓

Is SQL?

│        │

Yes      No

│         │

▼         ▼

SQL     General LLM
```

Example

```python
branch = RunnableBranch(
    ...
)
```

Common use cases:

- Router Agents
- Tool Routing
- Intent Classification

---

# 12. RunnablePassthrough

Sometimes you don't want to modify the input.

You simply forward it.

```text
Input

↓

RunnablePassthrough

↓

Same Input
```

Example

```python
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough()
```

Useful in RAG pipelines where the original question is needed later.

---

# 13. RunnableAssign

Adds new fields to existing data.

Example

Input

```python
{
    "question":"What is AI?"
}
```

After assign

```python
{
    "question":"What is AI?",
    "timestamp":"2026"
}
```

Useful for enriching data before sending it to the LLM.

---

# 14. Complete Runnable Ecosystem

```text
                     Runnable

                          │

    ┌──────────────┬───────────────┬───────────────┐

    ▼              ▼               ▼

Sequence      Parallel        Branch

    │              │               │

    ▼              ▼               ▼

Lambda      Passthrough       Assign
```

---

# 15. Chain vs Runnable

| Chain | Runnable |
|--------|----------|
| Workflow | Executable component |
| Multiple steps | Single building block |
| Built from Runnables | Basic unit |
| Represents the pipeline | Represents one stage |

Think of it this way:

```text
Runnable

↓

Building Block

↓

Chain

↓

Complete Workflow
```

---

# 16. Complete Execution Flow

```text
User

↓

Prompt Runnable

↓

LLM Runnable

↓

Parser Runnable

↓

Final Output
```

All together they form a **Chain**.

---

# 17. Enterprise Example

Customer asks:

```
Summarize this document.
```

Pipeline

```text
Question

↓

Retriever

↓

Prompt

↓

LLM

↓

Summary

↓

JSON Parser

↓

API Response
```

Every component is a Runnable.

The entire workflow is the Chain.

---

# 18. Best Practices

✅ Keep each Runnable focused on one responsibility.

✅ Build small reusable chains.

✅ Use RunnableParallel for independent tasks.

✅ Use RunnableBranch for routing decisions.

✅ Avoid creating one huge monolithic chain.

---

# 19. Common Mistakes

❌ Thinking Chains and Runnables are the same.

❌ Creating overly complex pipelines.

❌ Forgetting that every component must accept the previous component's output.

❌ Using parallel execution when tasks depend on each other.

---

# 20. Interview Questions

## Q1. What is a Chain?

**Answer:**

A Chain is a workflow where the output of one step becomes the input of the next step. Modern LangChain builds chains using LCEL rather than dedicated chain classes.

---

## Q2. What is a Runnable?

**Answer:**

A Runnable is any executable LangChain component that supports standardized execution methods such as invoke(), batch(), stream(), and ainvoke().

---

## Q3. What is the relationship between Chains and Runnables?

**Answer:**

A Chain is composed of one or more Runnables. Each Runnable performs a single task, while the Chain represents the complete workflow.

---

## Q4. What does the `|` operator create internally?

**Answer:**

It creates a `RunnableSequence`, where the output of one Runnable is passed as the input to the next Runnable.

---

## Q5. When would you use RunnableParallel?

**Answer:**

When multiple independent tasks can execute simultaneously, such as generating a summary and extracting keywords at the same time.

---

## Q6. When would you use RunnableBranch?

**Answer:**

For conditional routing, such as selecting different processing paths based on user intent or input type.

---

## Q7. What is RunnableLambda used for?

**Answer:**

It wraps custom Python functions so they can participate in LCEL pipelines as Runnables.

---

# 21. Quick Revision

| Component | Purpose |
|-----------|---------|
| Chain | Complete workflow |
| Runnable | Executable building block |
| RunnableSequence | Sequential execution |
| RunnableLambda | Custom Python logic |
| RunnableParallel | Parallel execution |
| RunnableBranch | Conditional routing |
| RunnablePassthrough | Forward input unchanged |
| RunnableAssign | Add new fields |

---

# Interview Cheat Sheet

```text
User

↓

Chain

↓

RunnableSequence

↓

Prompt

↓

LLM

↓

Parser

↓

Output

Other Runnable Types

RunnableLambda

RunnableParallel

RunnableBranch

RunnablePassthrough

RunnableAssign
```

---

# 30-Second Interview Answer

> **In modern LangChain, a Chain is a workflow built by composing Runnable components. Every PromptTemplate, Chat Model, Output Parser, or custom function is a Runnable that implements methods like `invoke()`, `batch()`, `stream()`, and `ainvoke()`. Using LCEL's `|` operator, these Runnables are connected into a `RunnableSequence`, creating modular, reusable, and production-ready AI pipelines.**

---

# Key Takeaway

> **Think of Runnables as LEGO blocks and Chains as the complete LEGO model. Every modern LangChain application is built by composing small, reusable Runnable components into powerful execution pipelines using LCEL.**